# RHINO FM Response & Frequency Axis Verification
**Author:** Jerry Mbulawa | **Supervisor:** Dr Phil Bull | **Collaborator:** Jordan Norris  
**Purpose:** Verify whether the suppressed FM band in RHINO wideband spectra is a real  
antenna/chain effect or a board artefact, and confirm the RFSoC frequency axis is correct.

## Tests in this notebook
| Cell | Test | Resolves |
|------|------|----------|
| 1 | 50 Ω terminator baseline | Board artefact vs FM pickup via cable |
| 2 | DIY stub notch (passive) | Frequency axis accuracy — no signal generator needed |
| 3 | Signal generator injection | Frequency axis accuracy + power calibration check |
| 4 | RSP1A SDR comparison | Independent FM visibility reference |
| 5 | FM band-stop filter | Quantify in-band leakage from FM block |
| 6 | Summary report | Structured pass/fail for all tests |

**Run Cell 0 first every session.** All results saved to `verification/` subdirectory.


In [12]:
# ================================================================
# CELL 0 — Setup: imports, hardware config, helper functions
# Run this first every session.
# ================================================================
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os, json, datetime, warnings
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────
OUT = Path('verification')
OUT.mkdir(exist_ok=True)
print(f'Output directory: {OUT.resolve()}')

# ── Hardware constants ───────────────────────────────────────────
FS_MHZ        = 4423.680        # ADC sample rate (MHz)
N_FFT         = 16384           # coarse FFT size
DF_MHZ        = FS_MHZ / N_FFT # channel width ≈ 0.270 MHz
RHINO_LO      = 60.0            # science band lower edge (MHz)
RHINO_HI      = 85.0            # science band upper edge (MHz)
FM_LO         = 87.5            # FM band lower edge (MHz)
FM_HI         = 108.0           # FM band upper edge (MHz)

# ADC safety: saturation onset ≈ 0 dBm at SMA (derived from balun
# geometry and loopback data — verify against DS926 Appendix D).
# Signal generator default is -20 dBm (20 dB below estimated saturation).
SIG_GEN_DBM   = -20.0           # TODO: confirm with Jordan on the day

# Stub notch parameters
VF_COAX       = 0.66            # velocity factor RG58
C_MS          = 299.792458e6    # speed of light (m/s)

# ── QICK init — comment out if running offline ────────────────────
HARDWARE_CONNECTED = False
ADC_CH = 0  # ADC channel used in v6 notebook
try:
    from qick import *
    from qick.averager_program import AveragerProgram
    soc = QickSoc()

    # ── DDR4 trigger programme (exact pattern from v6 Cell 5) ────
    # ddr4=True fires bit 13 on port 7 — required for live data.
    # Without this flag the DDR4 buffer returns stale zeros.
    class DDR4TriggerProgram(AveragerProgram):
        def initialize(self):
            self.declare_readout(ch=ADC_CH, length=1000, freq=0, gen_ch=None)
            self.synci(200)
        def body(self):
            self.trigger(adcs=[ADC_CH], ddr4=True, adc_trig_offset=100)
            self.wait_all()
            self.sync_all(self.us2cycles(1.0))

    _pcfg = {'ro_ch': ADC_CH, 'readout_length': 1000,
             'adc_trig_offset': 100, 'soft_avgs': 1,
             'reps': 1, 'relax_delay': 1.0}
    prog = DDR4TriggerProgram(soc, _pcfg)


    # Check DDC/LO configuration
    cfg_full = soc.get_cfg()
    ro_cfg = cfg_full['readouts'][ADC_CH]
    print(f"Readout channel {ADC_CH} config:")
    print(f"  fs (sampling rate): {ro_cfg.get('fs', 'NOT SET')} Hz")
    print(f"  adc_trig_offset: {ro_cfg.get('adc_trig_offset', 'NOT SET')}")
    print(f"  DDC/LO freq: {ro_cfg.get('lo_freq', 'NOT SET')} Hz")
    print(f"  mix_freq: {ro_cfg.get('mix_freq', 'NOT SET')} Hz")
    print(f"Full readout config keys: {ro_cfg.keys()}")


    HARDWARE_CONNECTED = True
    print(f'QICK connected. FS = {FS_MHZ:.3f} MHz')
except Exception as e:
    print(f'QICK not connected ({e}) — offline mode, data load only')

# ── DDR4 capture (exact pattern from v6 Cell 6) ──────────────────
# soc.readouts[ch].reset_buf()   — DOES NOT EXIST in QICK 0.2.388
# soc.readouts[ch].transfer_avg_buf() — DOES NOT EXIST
# Correct method: soc.ddr4_buf with set_switch / arm / get_mem
NT_COARSE    = 68   # transfers per coarse spectrum (N_FFT/256 + headroom)
SAMPLES_PER_XFER = 256

def _ddr4_capture_raw(nt):
    """Single DDR4 capture via soc.ddr4_buf (confirmed working on QICK 0.2.388)."""
    _cfg = soc.get_cfg()
    soc.ddr4_buf.set_switch(_cfg['readouts'][ADC_CH]['avgbuf_fullpath'])
    soc.clear_ddr4()
    soc.ddr4_buf.arm(nt=nt)
    prog.acquire(soc, load_pulses=False, progress=False)
    raw = soc.ddr4_buf.get_mem(nt=nt)
    return (raw[:, 0] if raw.ndim == 2 else raw).astype(np.float32)

# ── Helper: acquire one averaged spectrum ────────────────────────
def acquire_spectrum(n_frames=500, window='hann', label=''):
    """
    Acquire n_frames coarse FFT spectra and return (freq_mhz, power_db).
    Uses _ddr4_capture_raw — the only confirmed working path on this board.
    Requires HARDWARE_CONNECTED = True.
    """
    if not HARDWARE_CONNECTED:
        raise RuntimeError('Hardware not connected — cannot acquire')
    win  = np.hanning(N_FFT) if window == 'hann' else np.ones(N_FFT)
    enbw = 1.5 if window == 'hann' else 1.0
    acc  = np.zeros(N_FFT // 2 + 1)

    for _ in range(n_frames):
        raw     = _ddr4_capture_raw(NT_COARSE)
        I       = raw[:N_FFT].astype(np.float64)
        S       = np.fft.rfft(I * win)
        acc    += np.abs(S) ** 2

    spec    = acc / n_frames
    spec    = np.maximum(spec, 1e-30)
    spec_db = 10 * np.log10(spec) + 10 * np.log10(enbw)
    freq    = np.fft.rfftfreq(N_FFT) * FS_MHZ
    print(f'  Acquired {n_frames} frames — {label}')
    return freq, spec_db

# ── Helper: frequency → bin index ───────────────────────────────
def freq_to_bin(f_mhz):
    return int(round(f_mhz / DF_MHZ))

# ── Helper: save result dict ─────────────────────────────────────
def save_result(name, data):
    path = OUT / f'{name}.json'
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)
    print(f'  Saved: {path}')

# ── Result store (in-session) ────────────────────────────────────
RESULTS = {}

print('\nSetup complete.')
print(f'  Channel width: {DF_MHZ*1000:.1f} kHz/bin')
print(f'  Science band:  {RHINO_LO:.0f}–{RHINO_HI:.0f} MHz')
print(f'  FM band:       {FM_LO:.1f}–{FM_HI:.0f} MHz')
print(f'  Signal gen:    {SIG_GEN_DBM:.0f} dBm (default)')
print(f'  Hardware:      {"CONNECTED" if HARDWARE_CONNECTED else "OFFLINE"}')


Output directory: /home/xilinx/jupyter_notebooks/verification
Readout channel 0 config:
  fs (sampling rate): 4423.68 Hz
  adc_trig_offset: NOT SET
  DDC/LO freq: NOT SET Hz
  mix_freq: NOT SET Hz
Full readout config keys: dict_keys(['avg_maxlen', 'buf_maxlen', 'has_edge_counter', 'has_weights', 'trigger_type', 'trigger_port', 'trigger_bit', 'tproc_ch', 'adc', 'b_phase', 'fs', 'fs_mult', 'fs_div', 'decimation', 'f_fabric', 'f_dds', 'fdds_div', 'f_output', 'b_dds', 'iq_offset', 'has_outsel', 'avgbuf_revision', 'ro_revision', 'avgbuf_version', 'ro_version', 'avgbuf_type', 'ro_type', 'avgbuf_fullpath', 'ro_fullpath'])
QICK connected. FS = 4423.680 MHz

Setup complete.
  Channel width: 270.0 kHz/bin
  Science band:  60–85 MHz
  FM band:       87.5–108 MHz
  Signal gen:    -20 dBm (default)
  Hardware:      CONNECTED


In [8]:
ADC_CH = 0
_pcfg = {"ro_ch": ADC_CH, "readout_length": 1000,
"adc_trig_offset": 100, "soft_avgs": 1,
"reps": 1, "relax_delay": 1.0}
prog = DDR4TriggerProgram(soc, _pcfg)
print(f"Switched to ADC_CH = {ADC_CH} (ADC_D)")

Switched to ADC_CH = 0 (ADC_D)


In [9]:
import numpy as np
import matplotlib.pyplot as plt
_cfg = soc.get_cfg()
soc.ddr4_buf.set_switch(_cfg['readouts'][ADC_CH]['avgbuf_fullpath'])
soc.clear_ddr4()
soc.ddr4_buf.arm(nt=NT_COARSE)
prog.acquire(soc, load_pulses=False, progress=False)
raw = soc.ddr4_buf.get_mem(nt=NT_COARSE)

I_samples = (raw[:, 0] if raw.ndim == 2 else raw).astype(np.float32)
I_samples = I_samples[:N_FFT]

print(f"Raw I samples (ADC_D, post-power-cycle, after power cycle):")
print(f" Min: {I_samples.min():.2f}")
print(f" Max: {I_samples.max():.2f}")
print(f" Mean: {I_samples.mean():.4f}")
print(f" Std: {I_samples.std():.2f}")
print(f" RMS: {np.sqrt(np.mean(I_samples**2)):.2f}")
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(I_samples[:1000], lw=0.6)
ax.set_xlabel('Sample index'); ax.set_ylabel('ADC value')
ax.set_title('Raw time-domain samples, ADC_D, first 1000 points')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / 'jordan_step2_raw_timedomain.png', dpi=150)
plt.close(fig)
print("Saved: verification/jordan_step2_raw_timedomain.png")

Raw I samples (ADC_D, post-power-cycle, after power cycle):
 Min: -1167.00
 Max: 1156.00
 Mean: -0.3098
 Std: 817.36
 RMS: 817.36
Saved: verification/jordan_step2_raw_timedomain.png


In [10]:
ro_cfg = soc.get_cfg()['readouts'][ADC_CH]
print(f"--- Readout channel 1 (ADC_D) full config --POST-POWER-CYCLE---")
for k in ['fs', 'fs_mult', 'fs_div', 'decimation',
'f_fabric', 'f_dds', 'fdds_div', 'f_output', 'b_dds']:
    print(f" {k}: {ro_cfg.get(k, 'NOT SET')}")

--- Readout channel 1 (ADC_D) full config --POST-POWER-CYCLE---
 fs: 4423.68
 fs_mult: 18
 fs_div: 2
 decimation: 1
 f_fabric: 552.96
 f_dds: 4423.68
 fdds_div: 2
 f_output: 552.96
 b_dds: 32


In [11]:
freq_b, spec_b = acquire_spectrum(
200, label='B --ADC_D, 80 MHz, -10 dBm, before power cycle'
)
noise_floor = float(np.median(spec_b))
peak_idx = int(np.argmax(spec_b))
peak_freq = float(freq_b[peak_idx])
peak_power = float(spec_b[peak_idx])
peak_snr = peak_power - noise_floor

print(f"Noise floor: {noise_floor:.2f} dB")
print(f"Peak freq: {peak_freq:.3f} MHz")
print(f"Peak power: {peak_power:.2f} dB")
print(f"Peak SNR: {peak_snr:.2f} dB")
print(f"Frequency error (vs 80 MHz): {peak_freq - 80.0:+.3f} MHz")
# --- Empirically back out the true effective sample rate ---
N_bins = len(spec_b)
fs_effective_implied = 80.0 * N_FFT / peak_idx
scale_factor = FS_MHZ / fs_effective_implied
print(f"\n--- Empirical sample-rate back-out ---")
print(f"Peak bin index: {peak_idx} / {N_bins}")
print(f"Implied TRUE effective sample rate: {fs_effective_implied:.4f} MHz")
print(f"Currently assumed FS_MHZ: {FS_MHZ} MHz")
print(f"Scale factor (assumed / implied): {scale_factor:.4f}")

  Acquired 200 frames — B --ADC_D, 80 MHz, -10 dBm, before power cycle
Noise floor: 46.58 dB
Peak freq: 639.900 MHz
Peak power: 134.21 dB
Peak SNR: 87.63 dB
Frequency error (vs 80 MHz): +559.900 MHz

--- Empirical sample-rate back-out ---
Peak bin index: 2370 / 8193
Implied TRUE effective sample rate: 553.0464 MHz
Currently assumed FS_MHZ: 4423.68 MHz
Scale factor (assumed / implied): 7.9988


In [ ]:
ADC_CH = 0
_pcfg = {'ro_ch': ADC_CH, 'readout_length': 1000,
'adc_trig_offset': 100, 'soft_avgs': 1,
'reps': 1, 'relax_delay': 1.0}
prog = DDR4TriggerProgram(soc, _pcfg)
print(f"Switched to ADC_CH = {ADC_CH} (ADC_D)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
_cfg = soc.get_cfg()
soc.ddr4_buf.set_switch(_cfg['readouts'][ADC_CH]['avgbuf_fullpath'])
soc.clear_ddr4()
soc.ddr4_buf.arm(nt=NT_COARSE)
prog.acquire(soc, load_pulses=False, progress=False)
raw = soc.ddr4_buf.get_mem(nt=NT_COARSE)
I_samples = (raw[:, 0] if raw.ndim == 2 else raw).astype(np.float32)
I_samples = I_samples[:N_FFT]
print(f"Raw I samples (ADC_C, generator at 80 MHz, -10 dBm):")
print(f" Min: {I_samples.min():.2f}")
print(f" Max: {I_samples.max():.2f}")
print(f" Mean: {I_samples.mean():.4f}")
print(f" Std: {I_samples.std():.2f}")
print(f" RMS: {np.sqrt(np.mean(I_samples**2)):.2f}")
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(I_samples[:1000], lw=0.6)
ax.set_xlabel('Sample index'); ax.set_ylabel('ADC value')
ax.set_title('Raw time-domain samples, ADC_C, first 1000 points')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / 'jordan_step4_raw_timedomain_adcd.png', dpi=150)
plt.close(fig)
print("Saved: verification/jordan_step4_raw_timedomain_adcd.png")

In [ ]:
ro_cfg = soc.get_cfg()['readouts'][ADC_CH]
print(f"--- Readout channel {ADC_CH} (ADC_D) full config ---")
for k in ['fs', 'fs_mult', 'fs_div', 'decimation',
'f_fabric', 'f_dds', 'fdds_div', 'f_output', 'b_dds']:
    print(f" {k}: {ro_cfg.get(k, 'NOT SET')}")

In [ ]:
freq_test3, spec_test3 = acquire_spectrum(
200, label='Signal gen @ 80 MHz, -10 dBm, ADC D, Friday check'
)
noise_floor = float(np.median(spec_test3))
peak_idx = int(np.argmax(spec_test3))
peak_freq = float(freq_test3[peak_idx])
peak_power = float(spec_test3[peak_idx])
peak_snr = peak_power - noise_floor
print(f"\n=== TEST 4 RESULT (ADC_D, Friday) ===")
print(f"Noise floor: {noise_floor:.2f} dB")
print(f"Peak freq: {peak_freq:.3f} MHz")
print(f"Peak power: {peak_power:.2f} dB")
print(f"Peak SNR: {peak_snr:.2f} dB")
print(f"Frequency error (vs 80 MHz): {peak_freq - 80.0:+.3f} MHz")
# --- Empirically back out the true effective sample rate ---
N_bins = len(spec_test3)
fs_effective_implied = 80.0 * N_FFT / peak_idx
scale_factor = FS_MHZ / fs_effective_implied
print(f"\n--- Empirical sample-rate back-out ---")
print(f"Peak bin index: {peak_idx} / {N_bins}")
print(f"Implied TRUE effective sample rate: {fs_effective_implied:.4f} MHz")
print(f"Currently assumed FS_MHZ: {FS_MHZ} MHz")
print(f"Scale factor (assumed / implied): {scale_factor:.4f}")

In [ ]:
# Diagnostic: Check DDC/LO configuration
cfg_full = soc.get_cfg()
ro_cfg = cfg_full['readouts'][ADC_CH]

print(f"Readout channel {ADC_CH} (ADC_C) configuration:")
print(f"  fs (sampling rate): {ro_cfg.get('fs', 'NOT SET')}")
print(f"  adc_trig_offset: {ro_cfg.get('adc_trig_offset', 'NOT SET')}")
print(f"  DDC/LO freq: {ro_cfg.get('lo_freq', 'NOT SET')}")
print(f"  mix_freq: {ro_cfg.get('mix_freq', 'NOT SET')}")
print(f"\nFull readout config keys: {list(ro_cfg.keys())}")

In [ ]:
# Verify sampling rate interpretation
fs_mhz = 4423.68
fs_hz = fs_mhz * 1e6

# Expected bin spacing for coarse FFT (N=16384)
N_fft = 16384
bin_spacing_hz = fs_hz / N_fft
bin_spacing_mhz = bin_spacing_hz / 1e6

print(f"Sampling rate: {fs_mhz} MHz = {fs_hz:.2e} Hz")
print(f"FFT size: {N_fft}")
print(f"Bin spacing: {bin_spacing_hz:.2e} Hz = {bin_spacing_mhz:.4f} MHz")
print(f"Expected 80 MHz bin index: {80 / bin_spacing_mhz:.1f}")

# If fs were incorrectly treated as Hz (bug):
bin_spacing_wrong = (fs_mhz) / N_fft
print(f"\n--- IF BUG (fs in MHz treated as Hz) ---")
print(f"Wrong bin spacing: {bin_spacing_wrong:.6f} MHz")
print(f"Wrong 80 MHz maps to: {80 % bin_spacing_wrong:.3f} MHz (wraps around!)")

In [ ]:
# Capture raw time samples and inspect them directly
_cfg = soc.get_cfg()
soc.ddr4_buf.set_switch(_cfg['readouts'][ADC_CH]['avgbuf_fullpath'])
soc.clear_ddr4()
soc.ddr4_buf.arm(nt=NT_COARSE)
prog.acquire(soc, load_pulses=False, progress=False)
raw = soc.ddr4_buf.get_mem(nt=NT_COARSE)

# Extract I channel
I_samples = (raw[:, 0] if raw.ndim == 2 else raw).astype(np.float32)
I_samples = I_samples[:N_FFT]  # First 16384 samples

print(f"Raw time-domain I samples (first N_FFT={N_FFT}):")
print(f"  Min:  {I_samples.min():.2f}")
print(f"  Max:  {I_samples.max():.2f}")
print(f"  Mean: {I_samples.mean():.4f}  ← **THIS IS THE PROBLEM IF NON-ZERO**")
print(f"  Std:  {I_samples.std():.2f}")
print(f"  RMS:  {np.sqrt(np.mean(I_samples**2)):.2f}")

# Plot first 1000 samples to see the waveform
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(I_samples[:1000], lw=0.5)
ax.set_xlabel('Sample index')
ax.set_ylabel('ADC value')
ax.set_title('Raw Time-Domain Samples (first 1000 points) — Should show 80 MHz oscillation')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / 'raw_time_samples.png', dpi=150)
plt.close()
print("\nSaved: raw_time_samples.png")

# FFT of raw samples directly (sanity check)
fft_raw = np.fft.rfft(I_samples)
psd_raw = np.abs(fft_raw)**2 / N_FFT
spec_raw_db = 10 * np.log10(np.maximum(psd_raw, 1e-30))
freq_raw = np.fft.rfftfreq(N_FFT) * FS_MHZ

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(freq_raw, spec_raw_db, lw=0.8)
ax.axvline(80.0, color='red', ls='--', label='Expected: 80 MHz')
ax.set_xlabel('Frequency (MHz)')
ax.set_ylabel('Power (dB)')
ax.set_title('Direct FFT of Raw Time Samples')
ax.set_xlim(0, 200)
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / 'raw_fft_sanity_check.png', dpi=150)
plt.close()
print("Saved: raw_fft_sanity_check.png")

In [ ]:
# Test 3: Signal Generator at 80 MHz, -10 dBm
freq_test3, spec_test3 = acquire_spectrum(200, label='Signal gen @ 80 MHz, −10 dBm, ADC_C')

# Analyze the result
noise_floor = np.median(spec_test3)
peak_idx = np.argmax(spec_test3)
peak_freq = freq_test3[peak_idx]
peak_power = spec_test3[peak_idx]
peak_snr = peak_power - noise_floor

print(f"\n=== TEST 3 RESULT ===")
print(f"Noise floor: {noise_floor:.2f} dB")
print(f"Peak freq: {peak_freq:.3f} MHz")
print(f"Peak power: {peak_power:.2f} dB")
print(f"Peak SNR: {peak_snr:.2f} dB")
print(f"Frequency error: {peak_freq - 80.0:+.3f} MHz")

# Pass/fail criterion (from handover)
# Test 3: frequency error <1 bin (270 kHz), SNR ≥20 dB
PASS_FREQ_ERR = abs(peak_freq - 80.0) < 0.270
PASS_SNR = peak_snr >= 20.0

print(f"\nPass criteria:")
print(f"  Frequency error < 0.270 MHz: {PASS_FREQ_ERR}")
print(f"  SNR ≥ 20 dB: {PASS_SNR}")
print(f"  Overall: {'PASS ✓' if (PASS_FREQ_ERR and PASS_SNR) else 'FAIL ✗'}")

# Store result
RESULTS['test_3_signal_gen'] = {
    'generator_freq_mhz': 80.0,
    'generator_power_dbm': -10,
    'adc_channel': 'ADC_C',
    'n_frames': 200,
    'noise_floor_db': float(noise_floor),
    'peak_freq_mhz': float(peak_freq),
    'peak_power_db': float(peak_power),
    'peak_snr_db': float(peak_snr),
    'freq_error_mhz': float(peak_freq - 80.0),
    'passed': bool(PASS_FREQ_ERR and PASS_SNR)
}
save_result('test3_signal_gen_result', RESULTS['test_3_signal_gen'])

## Test 1 — 50 Ω Terminator
**What:** Replace the antenna input with a matched 50 Ω terminator. No sky, no cable, no FM path.

**Pass criterion:** The 100.1 MHz spur is absent or reduced by ≥ 10 dB compared to the baseline (no-terminator) spectrum.
- Spur drops ≥ 10 dB → **FM pickup via the cable/antenna** (not a board artefact)
- Spur persists within 3 dB → **Board-internal artefact**

**Before running:** Disconnect the antenna/cable and connect the 50 Ω terminator to the SMA input.

In [ ]:
# ================================================================
# CELL 1 — 50 Ω Terminator test
# BEFORE RUNNING: disconnect antenna, connect 50 Ω terminator to SMA.
# ================================================================
N_FRAMES_TERM = 500
SPUR_FREQ     = 100.1   # MHz — the known spur from CW calibration

print('Test 1: 50 Ω Terminator')
print('='*50)

if HARDWARE_CONNECTED:
    print(f'Acquiring {N_FRAMES_TERM} frames with 50 Ω terminator...')
    freq_t, spec_t = acquire_spectrum(N_FRAMES_TERM, label='50 Ohm terminator')
else:
    # ── Offline: load previously saved data ──────────────────────
    load_path = OUT / 'test1_terminator_spectrum.npz'
    if load_path.exists():
        d = np.load(load_path)
        freq_t, spec_t = d['freq'], d['spec']
        print(f'Loaded offline: {load_path}')
    else:
        print('No saved terminator data found. Run with hardware connected.')
        freq_t = np.fft.rfftfreq(N_FFT) * FS_MHZ
        spec_t = np.random.randn(len(freq_t)) * 2  # placeholder
        print('Using placeholder data for structure check only.')

# ── Save raw spectrum ────────────────────────────────────────────
np.savez(OUT / 'test1_terminator_spectrum.npz', freq=freq_t, spec=spec_t)

# ── Measure spur level ───────────────────────────────────────────
spur_bin   = freq_to_bin(SPUR_FREQ)
half_win   = 3  # search ± 3 bins around expected spur
spur_slice = spec_t[max(0, spur_bin-half_win) : spur_bin+half_win+1]
noise_mask = (freq_t >= 95) & (freq_t <= 105)
noise_mask[max(0,spur_bin-5):spur_bin+6] = False
noise_floor = float(np.median(spec_t[noise_mask]))
spur_level  = float(np.max(spur_slice))
spur_snr    = spur_level - noise_floor

# ── Load reference (prior no-terminator baseline if available) ───
ref_path = OUT / 'test1_reference_spectrum.npz'
has_ref  = ref_path.exists()
if has_ref:
    d_ref       = np.load(ref_path)
    spec_ref    = d_ref['spec']
    spur_ref    = float(np.max(spec_ref[max(0,spur_bin-half_win):spur_bin+half_win+1]))
    spur_drop   = spur_ref - spur_level
else:
    spur_drop   = None
    print('  No reference spectrum saved yet.')
    print('  To compare: first run with antenna connected and save as reference.')
    print('  (rename test1_terminator_spectrum.npz to test1_reference_spectrum.npz)')

# ── Pass/fail ────────────────────────────────────────────────────
PASS_THRESHOLD_DB = 10.0
if spur_drop is not None:
    passed = spur_drop >= PASS_THRESHOLD_DB
    verdict = 'PASS — FM pickup via cable/antenna' if passed else 'FAIL — likely board artefact'
else:
    passed  = None
    verdict = 'INCONCLUSIVE — no reference to compare against'

print(f'\n  Spur at {SPUR_FREQ} MHz:  {spur_level:.1f} dB')
print(f'  Local noise floor:     {noise_floor:.1f} dB')
print(f'  Spur SNR:              {spur_snr:.1f} dB above noise')
if spur_drop is not None:
    print(f'  Spur drop vs ref:      {spur_drop:.1f} dB')
print(f'  Verdict:               {verdict}')

RESULTS['test1_terminator'] = {
    'spur_freq_mhz': SPUR_FREQ,
    'spur_level_db': spur_level,
    'noise_floor_db': noise_floor,
    'spur_snr_db': spur_snr,
    'spur_drop_db': spur_drop,
    'pass_threshold_db': PASS_THRESHOLD_DB,
    'passed': passed,
    'verdict': verdict
}
save_result('test1_terminator', RESULTS['test1_terminator'])

# ── Plot ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
mask_plot = (freq_t >= 80) & (freq_t <= 120)
ax.plot(freq_t[mask_plot], spec_t[mask_plot],
        color='#1f77b4', lw=0.9, label='50 Ω terminator')
if has_ref:
    ax.plot(freq_t[mask_plot], spec_ref[mask_plot],
            color='#d62728', lw=0.9, alpha=0.7, label='Antenna (reference)')
ax.axvline(SPUR_FREQ, color='orange', lw=1.2, ls='--',
           label=f'Spur @ {SPUR_FREQ} MHz')
ax.axvspan(FM_LO, FM_HI, alpha=0.08, color='salmon', label='FM band')
ax.set_xlabel('Frequency (MHz)'); ax.set_ylabel('Power (dB)')
ax.set_title(f'Test 1 — 50 Ω Terminator | Verdict: {verdict}')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / 'test1_terminator.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'\n  Figure saved: {OUT}/test1_terminator.png')


## Test 2 — DIY Stub Notch (Passive Frequency Axis Check)
**What:** Cut a piece of RG58 coax to a quarter-wavelength at a chosen test frequency. Connect via a tee connector to the SMA input with the antenna attached. The open-ended stub creates a short circuit at the tee at the notch frequency, pulling that frequency to zero in the spectrum.

**No signal generator needed** — this is a purely passive test.

**Before running:** Calculate the required cable length below and cut the stub.

In [ ]:
# ================================================================
# CELL 2 — DIY Stub Notch — passive frequency axis verification
# BEFORE RUNNING:
#   1. Calculate stub length below
#   2. Cut RG58 coax to that length (leave one end open, put BNC/SMA on other)
#   3. Connect via tee to SMA input alongside antenna
# ================================================================

# ── Step 1: Calculate stub length ───────────────────────────────
# Choose a clean test frequency inside or near the science band.
# Avoid known RFI carriers. Good choices: 70.0, 75.0, 80.0 MHz
NOTCH_FREQ_MHZ = 75.0   # Hz — ADJUST to a quiet frequency in your spectrum

stub_length_m = (C_MS * VF_COAX) / (4 * NOTCH_FREQ_MHZ * 1e6)
stub_length_cm = stub_length_m * 100

print('Test 2: DIY Stub Notch — passive frequency axis check')
print('='*55)
print(f'\nTarget notch frequency:  {NOTCH_FREQ_MHZ:.1f} MHz')
print(f'Velocity factor (RG58):  {VF_COAX}')
print(f'Required stub length:    {stub_length_m*100:.1f} cm  ({stub_length_m*1000:.0f} mm)')
print(f'\nCut the RG58 to exactly {stub_length_cm:.1f} cm.')
print('Leave one end open-circuit (unterminated).')
print('Connect the other end to a tee on the SMA input.')
print('The notch should appear at {:.1f} MHz in the spectrum.'.format(NOTCH_FREQ_MHZ))
print('\nNow acquire the spectrum and run the analysis below.')

# ── Step 2: Acquire spectrum with stub ──────────────────────────
N_FRAMES_STUB = 500
NOTCH_SEARCH_BW = 3.0  # MHz — search window around expected notch

if HARDWARE_CONNECTED:
    print(f'\nAcquiring {N_FRAMES_STUB} frames with stub connected...')
    freq_s, spec_s = acquire_spectrum(N_FRAMES_STUB, label='stub notch')
else:
    load_path = OUT / 'test2_stub_spectrum.npz'
    if load_path.exists():
        d = np.load(load_path)
        freq_s, spec_s = d['freq'], d['spec']
        print(f'Loaded offline: {load_path}')
    else:
        print('No saved stub data found.')
        freq_s = np.fft.rfftfreq(N_FFT) * FS_MHZ
        spec_s = np.random.randn(len(freq_s)) * 2

np.savez(OUT / 'test2_stub_spectrum.npz', freq=freq_s, spec=spec_s)

# ── Step 3: Find the notch ───────────────────────────────────────
search_mask = (freq_s >= NOTCH_FREQ_MHZ - NOTCH_SEARCH_BW) &               (freq_s <= NOTCH_FREQ_MHZ + NOTCH_SEARCH_BW)
if not search_mask.any():
    raise ValueError(
        f'NOTCH_FREQ_MHZ={NOTCH_FREQ_MHZ} MHz is outside the spectrum '
        f'({freq_s.min():.1f}-{freq_s.max():.1f} MHz). Adjust NOTCH_FREQ_MHZ.')
notch_idx_local = np.argmin(spec_s[search_mask])
notch_freq_meas = float(freq_s[search_mask][notch_idx_local])
notch_level     = float(spec_s[search_mask][notch_idx_local])

# Surrounding noise floor (exclude notch region)
surround = (freq_s >= NOTCH_FREQ_MHZ - 8) & (freq_s <= NOTCH_FREQ_MHZ + 8)
surround[search_mask] = False
floor_surround = float(np.median(spec_s[surround])) if surround.any() else 0.0
notch_depth     = floor_surround - notch_level
freq_error_mhz  = abs(notch_freq_meas - NOTCH_FREQ_MHZ)
freq_error_bins = freq_error_mhz / DF_MHZ

# ── Pass criteria ────────────────────────────────────────────────
DEPTH_THRESHOLD  = 6.0   # dB — notch must be at least this deep to be real
ERROR_THRESHOLD  = 1.0   # bins — frequency axis error must be < 1 bin
passed_depth = notch_depth >= DEPTH_THRESHOLD
passed_freq  = freq_error_bins < ERROR_THRESHOLD
passed       = passed_depth and passed_freq
verdict      = 'PASS' if passed else 'FAIL'

print(f'\n  Expected notch: {NOTCH_FREQ_MHZ:.3f} MHz')
print(f'  Measured notch: {notch_freq_meas:.3f} MHz')
print(f'  Frequency error: {freq_error_mhz*1000:.0f} kHz  ({freq_error_bins:.2f} bins)')
print(f'  Notch depth:    {notch_depth:.1f} dB  (threshold: {DEPTH_THRESHOLD:.0f} dB)')
print(f'  Verdict:        {verdict}')
if not passed_depth:
    print('  NOTE: Notch too shallow — check stub connection or try a different frequency.')
if not passed_freq:
    print(f'  NOTE: Frequency error > 1 bin — check FS_MHZ constant matches actual sample rate.')

RESULTS['test2_stub'] = {
    'target_freq_mhz': NOTCH_FREQ_MHZ,
    'measured_freq_mhz': notch_freq_meas,
    'freq_error_mhz': freq_error_mhz,
    'freq_error_bins': freq_error_bins,
    'notch_depth_db': notch_depth,
    'stub_length_cm': stub_length_cm,
    'passed': passed,
    'verdict': verdict
}
save_result('test2_stub', RESULTS['test2_stub'])

# ── Plot ─────────────────────────────────────────────────────────
plot_lo = NOTCH_FREQ_MHZ - 10
plot_hi = NOTCH_FREQ_MHZ + 10
mask_p  = (freq_s >= plot_lo) & (freq_s <= plot_hi)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(freq_s[mask_p], spec_s[mask_p], color='#1f77b4', lw=1.0)
ax.axvline(NOTCH_FREQ_MHZ, color='orange', lw=1.2, ls='--',
           label=f'Expected notch: {NOTCH_FREQ_MHZ:.1f} MHz')
ax.axvline(notch_freq_meas, color='green', lw=1.2, ls=':',
           label=f'Measured notch: {notch_freq_meas:.3f} MHz (error: {freq_error_mhz*1000:.0f} kHz)')
ax.set_xlabel('Frequency (MHz)'); ax.set_ylabel('Power (dB)')
ax.set_title(f'Test 2 — Stub Notch @ {NOTCH_FREQ_MHZ:.1f} MHz | Depth: {notch_depth:.1f} dB | {verdict}')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / 'test2_stub_notch.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'\n  Figure saved: {OUT}/test2_stub_notch.png')


## Test 3 — Signal Generator Injection
**What:** Inject a CW tone at a known frequency and power level. Verify it appears at the correct bin and the power reading is consistent across frequencies.

**Before running:** Connect the signal generator output (via attenuator if needed) to the SMA input. Set output to `SIG_GEN_DBM` (default −20 dBm). Start with a conservative level and increase only if the tone is not clearly visible.

In [ ]:
# ================================================================
# CELL 3 — Signal generator injection — frequency axis + power check
# BEFORE RUNNING:
#   1. Connect signal generator to SMA input (via 20 dB attenuator if unsure)
#   2. Set output to SIG_GEN_DBM = -20 dBm (adjust below if needed)
#   3. Set generator to first TEST_FREQS entry
# ADC saturation threshold ≈ 0 dBm at SMA — stay well below this.
# ================================================================

# ── Test frequencies ─────────────────────────────────────────────
# Spread across science band and FM region for full coverage.
# Avoid known persistent RFI carriers (66, 67, 82 MHz).
TEST_FREQS_MHZ = [70.0, 80.0, 88.0, 100.0, 110.0]

# ── Generator power ───────────────────────────────────────────────
# Default -20 dBm: 20 dB below estimated ADC saturation.
# Increase to -10 dBm only if tone SNR < 20 dB.
# NEVER exceed -3 dBm without confirming DS926 absolute max.
SIG_GEN_DBM_TEST = SIG_GEN_DBM  # from Cell 0

N_FRAMES_SG = 200   # 200 frames sufficient for CW tone detection

print('Test 3: Signal Generator Injection')
print('='*50)
print(f'Test frequencies: {TEST_FREQS_MHZ} MHz')
print(f'Generator power:  {SIG_GEN_DBM_TEST:.0f} dBm')
print(f'SAFETY: do not exceed -3 dBm without checking DS926 Appendix D')
print()

tone_results = {}

for f_set in TEST_FREQS_MHZ:
    expected_bin = freq_to_bin(f_set)
    if HARDWARE_CONNECTED:
        input(f'  Set generator to {f_set:.1f} MHz at {SIG_GEN_DBM_TEST:.0f} dBm, then press Enter...')
    else:
        print(f'  [Offline] Simulating tone at {f_set:.1f} MHz')

    if HARDWARE_CONNECTED:
        freq_g, spec_g = acquire_spectrum(N_FRAMES_SG,
                                          label=f'CW @ {f_set:.1f} MHz')
    else:
        load_p = OUT / f'test3_cw_{f_set:.0f}MHz.npz'
        if load_p.exists():
            d = np.load(load_p)
            freq_g, spec_g = d['freq'], d['spec']
        else:
            freq_g  = np.fft.rfftfreq(N_FFT) * FS_MHZ
            spec_g  = np.random.randn(len(freq_g)) * 2

    np.savez(OUT / f'test3_cw_{f_set:.0f}MHz.npz', freq=freq_g, spec=spec_g)

    # Find peak in ±5 bins of expected location
    search_lo = max(0, expected_bin - 5)
    search_hi = min(len(spec_g)-1, expected_bin + 5)
    peak_idx  = np.argmax(spec_g[search_lo:search_hi]) + search_lo
    f_meas    = float(freq_g[peak_idx])
    f_err_mhz = abs(f_meas - f_set)
    f_err_bin = f_err_mhz / DF_MHZ

    # SNR
    noise_lo = max(0, expected_bin - 30)
    noise_hi = min(len(spec_g)-1, expected_bin + 30)
    noise_region = spec_g[noise_lo:noise_hi].copy()
    noise_region[expected_bin-noise_lo-8:expected_bin-noise_lo+9] = np.nan
    noise_floor_g = float(np.nanmedian(noise_region))
    snr_db        = float(spec_g[peak_idx]) - noise_floor_g

    passed_f   = f_err_bin < 1.0
    passed_snr = snr_db >= 20.0
    passed_t   = passed_f and passed_snr
    verdict_t  = 'PASS' if passed_t else 'FAIL'

    tone_results[f_set] = {
        'set_freq_mhz': f_set,
        'measured_freq_mhz': f_meas,
        'freq_error_mhz': f_err_mhz,
        'freq_error_bins': f_err_bin,
        'snr_db': snr_db,
        'passed': passed_t
    }
    print(f'  {f_set:6.1f} MHz → measured {f_meas:.3f} MHz  '
          f'(err {f_err_mhz*1000:.0f} kHz / {f_err_bin:.2f} bin)  '
          f'SNR {snr_db:.1f} dB  [{verdict_t}]')

# ── Overall pass/fail ─────────────────────────────────────────────
all_passed = all(v['passed'] for v in tone_results.values())
max_err_bin = max(v['freq_error_bins'] for v in tone_results.values())
print(f'\n  Overall verdict: {"PASS" if all_passed else "FAIL"}')
print(f'  Max frequency error: {max_err_bin:.2f} bins  (threshold < 1.0 bin)')

RESULTS['test3_siggen'] = {
    'tone_results': tone_results,
    'max_freq_error_bins': max_err_bin,
    'all_passed': all_passed
}
save_result('test3_siggen', RESULTS['test3_siggen'])

# ── Summary plot ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(TEST_FREQS_MHZ),
                         figsize=(3.5*len(TEST_FREQS_MHZ), 4))
for ax, f_set in zip(axes, TEST_FREQS_MHZ):
    load_p = OUT / f'test3_cw_{f_set:.0f}MHz.npz'
    if load_p.exists():
        d = np.load(load_p)
        freq_g2, spec_g2 = d['freq'], d['spec']
        b = freq_to_bin(f_set)
        win = 20
        sl  = slice(max(0,b-win), b+win+1)
        ax.plot(freq_g2[sl], spec_g2[sl], color='#1f77b4', lw=1.0)
        ax.axvline(f_set, color='orange', lw=1.0, ls='--')
        r = tone_results.get(f_set, {})
        verdict_t = 'PASS' if r.get('passed') else 'FAIL'
        ax.set_title(f'{f_set:.0f} MHz [{verdict_t}]\n'
                     f'err={r.get("freq_error_mhz",0)*1000:.0f} kHz  '
                     f'SNR={r.get("snr_db",0):.0f} dB',
                     fontsize=8)
        ax.set_xlabel('MHz'); ax.set_ylabel('dB')
        ax.grid(alpha=0.25)
fig.suptitle('Test 3 — Signal Generator Injection | Frequency Axis Check',
             fontsize=10)
fig.tight_layout()
fig.savefig(OUT / 'test3_siggen.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'\n  Figure saved: {OUT}/test3_siggen.png')


## Test 4 — RSP1A SDR Comparison
**What:** Compare the RFSoC spectrum against an RSP1A spectrum taken simultaneously with the same antenna. Divergence in the FM band localises any suppression to the RFSoC signal chain.

**SDR setup:** SDRuno or SDR-Console → export spectrum as CSV (Frequency Hz, Power dBFS). Save to `verification/rsp1a_spectrum.csv`.

In [ ]:
# ================================================================
# CELL 4 — RSP1A SDR comparison
# BEFORE RUNNING:
#   1. Connect RSP1A to same antenna as RFSoC (via power splitter), OR
#      swap antenna between both measurements (note any time difference)
#   2. In SDRuno/SDR-Console: set span to 55-120 MHz, match gain
#   3. Export spectrum CSV → save as verification/rsp1a_spectrum.csv
#      Expected format: two columns, Frequency (Hz) and Power (dBFS)
#      SDRuno: File → Save Spectrum → CSV
# ================================================================
import csv

RSP1A_CSV = OUT / 'rsp1a_spectrum.csv'
N_FRAMES_SDR = 500

print('Test 4: RSP1A SDR Comparison')
print('='*50)

# ── Acquire RFSoC spectrum ────────────────────────────────────────
if HARDWARE_CONNECTED:
    print(f'Acquiring {N_FRAMES_SDR} frames from RFSoC...')
    freq_r, spec_r = acquire_spectrum(N_FRAMES_SDR, label='RFSoC for SDR comparison')
    np.savez(OUT / 'test4_rfsoc_spectrum.npz', freq=freq_r, spec=spec_r)
else:
    load_p = OUT / 'test4_rfsoc_spectrum.npz'
    if load_p.exists():
        d = np.load(load_p)
        freq_r, spec_r = d['freq'], d['spec']
        print(f'Loaded offline: {load_p}')
    else:
        print('No saved RFSoC data. Run with hardware connected.')
        freq_r = np.fft.rfftfreq(N_FFT) * FS_MHZ
        spec_r = np.random.randn(len(freq_r)) * 2

# ── Load RSP1A CSV ────────────────────────────────────────────────
if not RSP1A_CSV.exists():
    print(f'\nRSP1A CSV not found at {RSP1A_CSV}')
    print('Export spectrum from SDRuno/SDR-Console and save to that path.')
    print('Skipping SDR comparison for now.')
    rsp_loaded = False
else:
    print(f'\nLoading RSP1A spectrum from {RSP1A_CSV}...')
    rsp_freq, rsp_power = [], []
    # Detect delimiter — SDRuno uses comma; some locales use semicolon
    with open(RSP1A_CSV, 'r') as f:
        sample = f.read(2048); f.seek(0)
        dialect = csv.Sniffer().sniff(sample, delimiters=',;\t')
        reader  = csv.reader(f, dialect)
        for row in reader:
            try:
                # Handle Hz or MHz input
                f_val = float(row[0].strip())
                p_val = float(row[1].strip())
                # Auto-detect units: Hz > 1e6, kHz > 1e3, else MHz
                if   f_val > 1e6: f_mhz = f_val / 1e6
                elif f_val > 1e3: f_mhz = f_val / 1e3
                else:             f_mhz = f_val
                rsp_freq.append(f_mhz)
                rsp_power.append(p_val)
            except (ValueError, IndexError):
                continue  # skip header rows

    rsp_freq  = np.array(rsp_freq)
    rsp_power = np.array(rsp_power)
    idx       = np.argsort(rsp_freq)
    rsp_freq, rsp_power = rsp_freq[idx], rsp_power[idx]
    print(f'  Loaded {len(rsp_freq)} points: {rsp_freq.min():.1f}–{rsp_freq.max():.1f} MHz')
    rsp_loaded = True

# ── Comparison analysis ───────────────────────────────────────────
if rsp_loaded:
    # Align offsets: normalise both spectra to their mean in the science band
    sci_mask_r = (freq_r >= RHINO_LO) & (freq_r <= RHINO_HI)
    sci_mask_s = (rsp_freq >= RHINO_LO) & (rsp_freq <= RHINO_HI)
    if not sci_mask_s.any():
        print('  WARN: RSP1A export does not cover the science band (60-85 MHz).')
        print('  Re-export from SDRuno with span covering 55-120 MHz.')
        rsp_loaded = False
    else:
        offset_r = float(np.mean(spec_r[sci_mask_r]))
        offset_s = float(np.mean(rsp_power[sci_mask_s]))
    spec_r_n   = spec_r - offset_r     # normalised RFSoC
    rsp_n      = rsp_power - offset_s  # normalised RSP1A

    # FM band level in each receiver (after normalisation)
    fm_mask_r  = (freq_r >= FM_LO) & (freq_r <= FM_HI)
    fm_mask_s  = (rsp_freq >= FM_LO) & (rsp_freq <= FM_HI)
    fm_rfsoc   = float(np.mean(spec_r_n[fm_mask_r]))
    fm_rsp1a   = float(np.mean(rsp_n[fm_mask_s]))
    fm_diff    = fm_rfsoc - fm_rsp1a  # negative = RFSoC sees less FM

    SUPPRESS_THRESHOLD = -5.0  # dB — RFSoC < RSP1A by > 5 dB = suppression
    suppression_confirmed = fm_diff < SUPPRESS_THRESHOLD
    verdict = ('RFSoC FM suppression CONFIRMED — RFSoC sees {:.1f} dB less FM than RSP1A'
               .format(abs(fm_diff)) if suppression_confirmed else
               'No significant FM suppression — levels match within {:.1f} dB'
               .format(abs(fm_diff)))
    print(f'\n  FM band (normalised):')
    print(f'    RFSoC: {fm_rfsoc:+.1f} dB rel. to science band mean')
    print(f'    RSP1A: {fm_rsp1a:+.1f} dB rel. to science band mean')
    print(f'    Difference (RFSoC − RSP1A): {fm_diff:+.1f} dB')
    print(f'  Verdict: {verdict}')

    RESULTS['test4_sdr'] = {
        'fm_level_rfsoc_db': fm_rfsoc,
        'fm_level_rsp1a_db': fm_rsp1a,
        'fm_difference_db': fm_diff,
        'suppression_confirmed': suppression_confirmed,
        'verdict': verdict
    }
    save_result('test4_sdr', RESULTS['test4_sdr'])

    # ── Plot ─────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 4))
    m55_120 = (freq_r >= 55) & (freq_r <= 120)
    ax.plot(freq_r[m55_120], spec_r_n[m55_120],
            color='#1f77b4', lw=1.0, label='RFSoC (normalised)')
    m55_120s = (rsp_freq >= 55) & (rsp_freq <= 120)
    ax.plot(rsp_freq[m55_120s], rsp_n[m55_120s],
            color='#d62728', lw=1.0, alpha=0.8, label='RSP1A (normalised)')
    ax.axvspan(RHINO_LO, RHINO_HI, alpha=0.08, color='gold',
               label='Science band')
    ax.axvspan(FM_LO, FM_HI, alpha=0.08, color='salmon', label='FM band')
    ax.set_xlabel('Frequency (MHz)'); ax.set_ylabel('Normalised power (dB)')
    ax.set_title(f'Test 4 — RSP1A vs RFSoC | FM diff: {fm_diff:+.1f} dB | {"CONFIRMED" if suppression_confirmed else "NOT CONFIRMED"}')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / 'test4_sdr_comparison.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'\n  Figure saved: {OUT}/test4_sdr_comparison.png')


## Test 5 — FM Band-Stop Filter
**What:** Acquire spectra with and without the FM band-stop filter in the chain. Quantify how much FM power is attenuated in the FM band and confirm the science band is not significantly affected.

**Before running:** First acquire without filter (antenna only), then insert the FM band-stop filter between antenna and SMA.

In [ ]:
# ================================================================
# CELL 5 — FM band-stop filter characterisation
# BEFORE RUNNING:
#   Step A: acquire WITHOUT filter (antenna only) → press Enter
#   Step B: insert FM band-stop filter → press Enter to acquire again
# ================================================================
N_FRAMES_FILT = 500

print('Test 5: FM Band-Stop Filter Characterisation')
print('='*55)

# ── Step A: without filter ────────────────────────────────────────
if HARDWARE_CONNECTED:
    input('Step A: Connect ANTENNA ONLY (no filter). Press Enter to acquire...')
if HARDWARE_CONNECTED:
    freq_f, spec_nofilt = acquire_spectrum(N_FRAMES_FILT, label='no filter')
    np.savez(OUT / 'test5_no_filter.npz', freq=freq_f, spec=spec_nofilt)
else:
    load_p = OUT / 'test5_no_filter.npz'
    freq_f = np.fft.rfftfreq(N_FFT) * FS_MHZ
    if load_p.exists():
        spec_nofilt = np.load(load_p)['spec']
        print('Loaded offline: no-filter spectrum')
    else:
        spec_nofilt = np.random.randn(len(freq_f)) * 2

# ── Step B: with filter ───────────────────────────────────────────
if HARDWARE_CONNECTED:
    input('Step B: Insert FM band-stop filter between antenna and SMA. Press Enter...')
if HARDWARE_CONNECTED:
    _, spec_filt = acquire_spectrum(N_FRAMES_FILT, label='with FM band-stop filter')
    np.savez(OUT / 'test5_with_filter.npz', freq=freq_f, spec=spec_filt)
else:
    load_p2 = OUT / 'test5_with_filter.npz'
    if load_p2.exists():
        spec_filt = np.load(load_p2)['spec']
        print('Loaded offline: filter spectrum')
    else:
        spec_filt = spec_nofilt - np.where(
            (freq_f >= FM_LO) & (freq_f <= FM_HI), 25.0, 0.5)

# ── Measure attenuation ───────────────────────────────────────────
insertion_loss = spec_filt - spec_nofilt  # negative = filter attenuates

sci_mask = (freq_f >= RHINO_LO) & (freq_f <= RHINO_HI)
fm_mask  = (freq_f >= FM_LO)    & (freq_f <= FM_HI)

sci_loss = float(np.mean(insertion_loss[sci_mask]))
fm_att   = float(-np.mean(insertion_loss[fm_mask]))  # positive = attenuation

# Pass criteria
SCI_LOSS_MAX = 1.0   # dB — science band loss must be < 1 dB
FM_ATT_MIN   = 20.0  # dB — FM attenuation must be > 20 dB
passed = (abs(sci_loss) < SCI_LOSS_MAX) and (fm_att > FM_ATT_MIN)
verdict = 'PASS' if passed else 'FAIL'

print(f'\n  Science band insertion loss: {sci_loss:.2f} dB  (threshold < {SCI_LOSS_MAX} dB)')
print(f'  FM band attenuation:        {fm_att:.1f} dB   (threshold > {FM_ATT_MIN} dB)')
print(f'  Verdict: {verdict}')

RESULTS['test5_filter'] = {
    'science_band_loss_db': sci_loss,
    'fm_attenuation_db': fm_att,
    'passed': passed,
    'verdict': verdict
}
save_result('test5_filter', RESULTS['test5_filter'])

# ── Plot ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
mask_plot = (freq_f >= 55) & (freq_f <= 120)
axes[0].plot(freq_f[mask_plot], spec_nofilt[mask_plot],
             color='#d62728', lw=1.0, label='No filter')
axes[0].plot(freq_f[mask_plot], spec_filt[mask_plot],
             color='#1f77b4', lw=1.0, label='FM band-stop filter')
axes[0].axvspan(RHINO_LO, RHINO_HI, alpha=0.08, color='gold')
axes[0].axvspan(FM_LO, FM_HI, alpha=0.08, color='salmon')
axes[0].set_xlabel('Frequency (MHz)'); axes[0].set_ylabel('Power (dB)')
axes[0].set_title('Spectra with and without filter'); axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(freq_f[mask_plot], -insertion_loss[mask_plot],
             color='#2ca02c', lw=1.0, label='Filter attenuation')
axes[1].axhline(0, color='grey', lw=0.7, ls='--')
axes[1].axvspan(RHINO_LO, RHINO_HI, alpha=0.08, color='gold',
                label=f'Science band (loss={sci_loss:.2f} dB)')
axes[1].axvspan(FM_LO, FM_HI, alpha=0.08, color='salmon',
                label=f'FM band (atten={fm_att:.1f} dB)')
axes[1].set_xlabel('Frequency (MHz)'); axes[1].set_ylabel('Attenuation (dB)')
axes[1].set_title(f'Filter attenuation profile | {verdict}')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / 'test5_filter.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'\n  Figure saved: {OUT}/test5_filter.png')


## Test 6 — Summary Report
Run after completing all tests. Generates a structured pass/fail table and saves a JSON report.

In [ ]:
# ================================================================
# CELL 6 — Verification summary report
# Run after completing all tests.
# ================================================================
import datetime

print('RHINO FM Response & Frequency Axis Verification — Summary')
print('='*62)
print(f'Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}')
print()

rows = [
    ('Test 1', '50 Ω Terminator',       'test1_terminator', 'verdict'),
    ('Test 2', 'Stub notch',             'test2_stub',       'verdict'),
    ('Test 3', 'Signal generator',       'test3_siggen',     'all_passed'),
    ('Test 4', 'RSP1A SDR comparison',   'test4_sdr',        'suppression_confirmed'),
    ('Test 5', 'FM band-stop filter',    'test5_filter',     'verdict'),
]

summary = {}
for code_str, name, key, field in rows:
    # Load from file if not in session
    if key not in RESULTS:
        fp = OUT / f'{key}.json'
        if fp.exists():
            with open(fp) as f:
                RESULTS[key] = json.load(f)
    result = RESULTS.get(key, {})
    val    = result.get(field, 'NOT RUN')
    if isinstance(val, bool):
        status = 'PASS' if val else 'FAIL'
    else:
        status = str(val) if val != 'NOT RUN' else 'NOT RUN'
    summary[code_str] = status
    print(f'  {code_str}: {name:<25} {status}')

print()

# ── Key numbers ───────────────────────────────────────────────────
if 'test1_terminator' in RESULTS:
    r = RESULTS['test1_terminator']
    print(f'  Spur drop (terminator vs antenna): '
          f'{r.get("spur_drop_db", "N/A")} dB')
if 'test2_stub' in RESULTS:
    r = RESULTS['test2_stub']
    print(f'  Stub notch frequency error:        '
          f'{r.get("freq_error_mhz",0)*1000:.0f} kHz  '
          f'({r.get("freq_error_bins",0):.2f} bins)')
if 'test3_siggen' in RESULTS:
    r = RESULTS['test3_siggen']
    print(f'  Max tone frequency error:          '
          f'{r.get("max_freq_error_bins", "N/A"):.2f} bins')
if 'test4_sdr' in RESULTS:
    r = RESULTS['test4_sdr']
    print(f'  RFSoC vs RSP1A FM difference:      '
          f'{r.get("fm_difference_db", "N/A"):.1f} dB')
if 'test5_filter' in RESULTS:
    r = RESULTS['test5_filter']
    print(f'  FM band-stop attenuation:          '
          f'{r.get("fm_attenuation_db", "N/A"):.1f} dB')
    print(f'  Science band insertion loss:        '
          f'{r.get("science_band_loss_db", "N/A"):.2f} dB')

# ── Save full report ─────────────────────────────────────────────
report = {
    'generated': datetime.datetime.now().isoformat(),
    'summary': summary,
    'results': RESULTS
}
with open(OUT / 'verification_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print(f'\n  Full report saved: {OUT}/verification_report.json')
all_done = all(v != 'NOT RUN' for v in summary.values())
all_pass = all(v in ('PASS', 'True') for v in summary.values() if v != 'NOT RUN')
print(f'\n  Overall: {"ALL PASS" if all_pass and all_done else "INCOMPLETE OR FAILURES — see above"}')


In [ ]:
ADC_CH = 1

_pcfg = {'ro_ch': ADC_CH, 'readout_length': 1000,
         'adc_trig_offset': 100, 'soft_avgs': 1,
         'reps': 1, 'relax_delay': 1.0}
prog = DDR4TriggerProgram(soc, _pcfg)

print(f"Switched to ADC_CH = {ADC_CH} (ADC_C)")
print("Now move the SMA cable from ADC_D to ADC_C before running the next capture.")

In [ ]:
# Wideband check while generator outputs CW at 80 MHz — text output only
freq_check, spec_check = acquire_spectrum(200, label='wideband check @ 80MHz nominal- ADC_C')

import numpy as np

plot_mask = freq_check <= 200  # MHz, wide view
f = freq_check[plot_mask]
s = spec_check[plot_mask]

# ── Noise floor and global peak ──────────────────────────────────
noise_floor = float(np.median(s))
peak_i = int(np.argmax(s))
peak_freq = float(f[peak_i])
peak_level = float(s[peak_i])
peak_snr = peak_level - noise_floor

print(f"Noise floor (median, 0-200 MHz): {noise_floor:.1f} dB")
print(f"Strongest peak overall:          {peak_freq:.3f} MHz  at {peak_level:.1f} dB  (SNR {peak_snr:.1f} dB)")
print(f"Distance from 80.0 MHz:          {peak_freq - 80.0:+.3f} MHz")
print()

# ── Top 10 local-maximum peaks, ranked by SNR ────────────────────
THRESH_DB = 6.0   # minimum SNR above noise floor to count as a peak
is_peak = np.zeros(len(s), dtype=bool)
for i in range(1, len(s) - 1):
    if s[i] > s[i-1] and s[i] > s[i+1] and (s[i] - noise_floor) > THRESH_DB:
        is_peak[i] = True

peak_idxs = np.where(is_peak)[0]
# Suppress peaks within 0.5 MHz of a stronger neighbour
peak_idxs = peak_idxs[np.argsort(-s[peak_idxs])]  # sort by strength, strongest first
kept = []
for idx in peak_idxs:
    if all(abs(f[idx] - f[k]) > 0.5 for k in kept):
        kept.append(idx)
kept = sorted(kept, key=lambda i: -s[i])[:10]

print(f"Top peaks found (SNR > {THRESH_DB:.0f} dB above noise floor):")
print(f"{'Freq (MHz)':>12} {'Power (dB)':>12} {'SNR (dB)':>10} {'Δ from 80MHz':>14}")
print("-" * 52)
if kept:
    for idx in kept:
        print(f"{f[idx]:>12.3f} {s[idx]:>12.1f} {s[idx]-noise_floor:>10.1f} {f[idx]-80.0:>+14.3f}")
else:
    print("  (none found — spectrum is flat, no signal above threshold anywhere)")

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

plot_mask = freq_check <= 200
f = freq_check[plot_mask]
s = spec_check[plot_mask]

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Full view
axes[0].plot(f, s, lw=0.8, color='#1f77b4')
axes[0].axvline(100.0, color='red', ls='--', label='Generator set to 100.0 MHz')
axes[0].set_xlabel('Frequency (MHz)'); axes[0].set_ylabel('Power (dB)')
axes[0].set_title('Full view: 0-200 MHz')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Zoomed view around 80 MHz
zoom_mask = (freq_check >= 70) & (freq_check <= 90)
axes[1].plot(freq_check[zoom_mask], spec_check[zoom_mask], lw=1.0, color='darkorange')
axes[1].axvline(80.0, color='red', ls='--', label='Expected: 80.0 MHz')
axes[1].set_xlabel('Frequency (MHz)'); axes[1].set_ylabel('Power (dB)')
axes[1].set_title('Zoomed view: 70-90 MHz')
axes[1].legend(); axes[1].grid(alpha=0.3)

fig.tight_layout()
out_path = 'verification/wideband_check_neg10dbm_adcC.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.close(fig)

display(Image(filename=out_path))
print(f"Saved: {out_path}")

In [ ]:
try:
    cur_atten = soc.get_adc_attenuator('00')
    print(f"Current ADC attenuator: {cur_atten} dB")
except Exception as e:
    print(f"Could not read attenuator: {e}")

In [13]:
# ============================================================
# RAW IQ OBSERVATION — antenna + FM band-stop filter in chain
# Run this cell to set parameters and get a pre-flight checklist.
# Run Cell 1 (setup) first if you haven't already this session.
# ============================================================

# ---- EDIT THESE BEFORE RUNNING ----
ANTENNA_NAME      = "discone"  # e.g. "whip", "horn"
LNA_IN_CHAIN      = True   # matches Test 5's chain (LNA + FM filter)
FM_FILTER_IN_CHAIN = True  # should be True for this observation — confirm physically!
ADC_CH_OBS        = 0      # 0 = ADC_D (default, matches prior antenna tests), 1 = ADC_C

N_FRAMES          = 1500   # ~number of snapshots to capture; see timing note below
CHECKPOINT_EVERY  = 200    # write partial data to disk every N frames, in case of a crash/disconnect

OUT_IQ = Path('iq_observations')
OUT_IQ.mkdir(exist_ok=True)

print("="*60)
print("PRE-FLIGHT CHECKLIST — confirm all of these physically before running Cell B")
print("="*60)
print(f"  [ ] Antenna connected: {ANTENNA_NAME}")
print(f"  [ ] LNA in chain:      {LNA_IN_CHAIN}")
print(f"  [ ] FM band-stop filter in chain: {FM_FILTER_IN_CHAIN}")
print(f"  [ ] Cable plugged into ADC_{'D' if ADC_CH_OBS==0 else 'C'} "
      f"(ADC_CH_OBS = {ADC_CH_OBS}) — does this match where it's physically plugged in?")
print(f"  [ ] Signal generator OUTPUT IS OFF (this is a sky observation, not a cal tone)")
print("="*60)

if ANTENNA_NAME.startswith("TODO"):
    print("\n*** STOP: set ANTENNA_NAME before continuing. ***")

PRE-FLIGHT CHECKLIST — confirm all of these physically before running Cell B
  [ ] Antenna connected: discone
  [ ] LNA in chain:      True
  [ ] FM band-stop filter in chain: True
  [ ] Cable plugged into ADC_D (ADC_CH_OBS = 0) — does this match where it's physically plugged in?
  [ ] Signal generator OUTPUT IS OFF (this is a sky observation, not a cal tone)


In [14]:
# ============================================================
# Raw IQ capture loop. Saves uncalibrated ADC counts + full
# readout config, so frequency axis can be correctly assigned
# later regardless of the 8x/10x scale-factor question.
# ============================================================
import time

assert HARDWARE_CONNECTED, "Hardware not connected — re-run Cell 1 setup first"
assert not ANTENNA_NAME.startswith("TODO"), "Fill in ANTENNA_NAME in Cell A first"

# Select the channel for this observation (rebuilds prog/ADC_CH, same pattern as the
# Test 3 channel-switch cells — does this even if you think you're already on the right channel)
ADC_CH = ADC_CH_OBS
_pcfg = {'ro_ch': ADC_CH, 'readout_length': 1000,
         'adc_trig_offset': 100, 'soft_avgs': 1,
         'reps': 1, 'relax_delay': 1.0}
prog = DDR4TriggerProgram(soc, _pcfg)
print(f"Capturing on ADC_CH = {ADC_CH} ({'ADC_D' if ADC_CH==0 else 'ADC_C'})")

def _ddr4_capture_raw_iq(nt):
    """Like _ddr4_capture_raw(), but keeps BOTH columns instead of truncating to I only."""
    _cfg = soc.get_cfg()
    soc.ddr4_buf.set_switch(_cfg['readouts'][ADC_CH]['avgbuf_fullpath'])
    soc.clear_ddr4()
    soc.ddr4_buf.arm(nt=nt)
    prog.acquire(soc, load_pulses=False, progress=False)
    raw = soc.ddr4_buf.get_mem(nt=nt)
    return raw.astype(np.float32)  # shape (n_samples, 2) if Q channel is present, else (n_samples,)

# Full config dump — this is the ground truth needed to correctly calibrate
# frequency later, whatever the 8x/10x question resolves to.
ro_cfg = soc.get_cfg()['readouts'][ADC_CH]
cfg_dump = {k: ro_cfg.get(k, None) for k in
            ['fs', 'fs_mult', 'fs_div', 'decimation', 'f_fabric', 'f_dds', 'fdds_div', 'f_output', 'b_dds']}
print("Readout config at capture time:")
for k, v in cfg_dump.items():
    print(f"  {k}: {v}")

frames = []
timestamps = []
t_start = time.time()

print(f"\nCapturing {N_FRAMES} frames...")
try:
    for i in range(N_FRAMES):
        frame = _ddr4_capture_raw_iq(NT_COARSE)
        frames.append(frame)
        timestamps.append(time.time())

        if (i + 1) % 50 == 0:
            elapsed = time.time() - t_start
            print(f"  frame {i+1}/{N_FRAMES}  ({elapsed:.1f}s elapsed, "
                  f"{elapsed/(i+1):.3f}s/frame)")

        if (i + 1) % CHECKPOINT_EVERY == 0:
            ts_str = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            np.savez(OUT_IQ / f'iq_checkpoint_{ts_str}.npz',
                     frames=np.array(frames), timestamps=np.array(timestamps),
                     **cfg_dump, adc_ch=ADC_CH, antenna=ANTENNA_NAME,
                     lna_in_chain=LNA_IN_CHAIN, fm_filter_in_chain=FM_FILTER_IN_CHAIN,
                     fs_mhz_assumed=FS_MHZ, nt_coarse=NT_COARSE,
                     samples_per_xfer=SAMPLES_PER_XFER)
            print(f"    [checkpoint saved: {i+1} frames]")

except KeyboardInterrupt:
    print(f"\nInterrupted after {len(frames)} frames — saving what we have.")

t_end = time.time()

# Final save
ts_str = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
final_path = OUT_IQ / f'iq_observation_{ANTENNA_NAME}_{ts_str}.npz'
np.savez(final_path,
         frames=np.array(frames), timestamps=np.array(timestamps),
         **cfg_dump, adc_ch=ADC_CH, antenna=ANTENNA_NAME,
         lna_in_chain=LNA_IN_CHAIN, fm_filter_in_chain=FM_FILTER_IN_CHAIN,
         fs_mhz_assumed=FS_MHZ, nt_coarse=NT_COARSE,
         samples_per_xfer=SAMPLES_PER_XFER,
         capture_start=t_start, capture_end=t_end)

print(f"\nDone. {len(frames)} frames saved to: {final_path}")
print(f"Total duration: {t_end - t_start:.1f}s, "
      f"file size: {final_path.stat().st_size / 1e6:.1f} MB")

Capturing on ADC_CH = 0 (ADC_D)
Readout config at capture time:
  fs: 4423.68
  fs_mult: 18
  fs_div: 2
  decimation: 1
  f_fabric: 552.96
  f_dds: 4423.68
  fdds_div: 2
  f_output: 552.96
  b_dds: 32

Capturing 1500 frames...
  frame 50/1500  (233.4s elapsed, 4.667s/frame)
  frame 100/1500  (465.9s elapsed, 4.659s/frame)
  frame 150/1500  (698.4s elapsed, 4.656s/frame)
  frame 200/1500  (931.0s elapsed, 4.655s/frame)
    [checkpoint saved: 200 frames]
  frame 250/1500  (1163.8s elapsed, 4.655s/frame)
  frame 300/1500  (1396.3s elapsed, 4.654s/frame)
  frame 350/1500  (1628.9s elapsed, 4.654s/frame)
  frame 400/1500  (1861.4s elapsed, 4.654s/frame)
    [checkpoint saved: 400 frames]
  frame 450/1500  (2094.5s elapsed, 4.654s/frame)
  frame 500/1500  (2327.0s elapsed, 4.654s/frame)
  frame 550/1500  (2559.6s elapsed, 4.654s/frame)
  frame 600/1500  (2792.2s elapsed, 4.654s/frame)
    [checkpoint saved: 600 frames]
  frame 650/1500  (3025.6s elapsed, 4.655s/frame)
  frame 700/1500  (3258

In [16]:
# Quick sanity plot only — confirms signal is present before you leave the bench.
# Frequency axis here uses the CURRENTLY ASSUMED FS_MHZ, which is unverified —
# do not treat this plot's frequency labels as trustworthy for real analysis later.

frame0 = frames[0]
I0 = frame0[:, 0] if frame0.ndim == 2 else frame0

fig, axes = plt.subplots(2, 1, figsize=(12, 7))
axes[0].plot(I0[:1000], lw=0.6)
axes[0].set_title(f'First frame, first 1000 raw samples — {ANTENNA_NAME}, '
                   f'FM filter {"IN" if FM_FILTER_IN_CHAIN else "OUT"}')
axes[0].set_xlabel('Sample index'); axes[0].set_ylabel('ADC counts')
axes[0].grid(alpha=0.3)

spec = np.abs(np.fft.rfft(I0.astype(np.float64) * np.hanning(len(I0))))**2
freq_provisional = np.fft.rfftfreq(len(I0)) * FS_MHZ  # PROVISIONAL — see warning above
axes[1].plot(freq_provisional, 10*np.log10(spec + 1e-30), lw=0.6)
axes[1].set_title('Single-frame FFT — frequency axis PROVISIONAL, do not trust for analysis')
axes[1].set_xlabel('Frequency (MHz, provisional)'); axes[1].set_ylabel('Power (dB)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_IQ / 'observation_sanity_check.png', dpi=120)
plt.show()

print(f"RMS (raw counts): {np.sqrt(np.mean(I0.astype(np.float64)**2)):.2f}")
print("If this is in the single digits, the antenna/cable likely isn't connected properly.")

RMS (raw counts): 85.78
If this is in the single digits, the antenna/cable likely isn't connected properly.


In [18]:
# ============================================================
# 500 MHz ALIASING TEST — discriminates between scale-factor hypotheses
# Recommend: LNA and FM filter BYPASSED for this test (see note above)
# ============================================================

print("="*60)
print("PRE-FLIGHT — confirm before running")
print("="*60)
print("  [ ] Signal generator set to 500.000000 MHz, -10 dBm, OUTPUT ON")
print("  [ ] LNA bypassed (recommended) — confirm with Jordan if unsure")
print("  [ ] FM band-stop filter bypassed (recommended)")
print("  [ ] Cable plugged into ADC_D (or note if using ADC_C instead)")
print("="*60)

ADC_CH = 0  # ADC_D, matching the discone session — change to 1 if using ADC_C
_pcfg = {'ro_ch': ADC_CH, 'readout_length': 1000,
         'adc_trig_offset': 100, 'soft_avgs': 1,
         'reps': 1, 'relax_delay': 1.0}
prog = DDR4TriggerProgram(soc, _pcfg)
print(f"\nReady on ADC_CH = {ADC_CH}")

# Predicted outcomes -- computed BEFORE capture so you can read the result instantly
predictions = {
    "H1: no decimation (code already correct)": 500.04,
    "H2: 8x  (f_output = 552.96 MHz)":          423.63,
    "H3: 10x (FS_MHZ/10 = 442.368 MHz)":         576.45,
}
print("\nPredicted peak location on the CURRENT (uncorrected) frequency axis:")
for name, pred in predictions.items():
    print(f"  {name:45s} -> ~{pred:.1f} MHz")

PRE-FLIGHT — confirm before running
  [ ] Signal generator set to 500.000000 MHz, -10 dBm, OUTPUT ON
  [ ] LNA bypassed (recommended) — confirm with Jordan if unsure
  [ ] FM band-stop filter bypassed (recommended)
  [ ] Cable plugged into ADC_D (or note if using ADC_C instead)

Ready on ADC_CH = 0

Predicted peak location on the CURRENT (uncorrected) frequency axis:
  H1: no decimation (code already correct)      -> ~500.0 MHz
  H2: 8x  (f_output = 552.96 MHz)               -> ~423.6 MHz
  H3: 10x (FS_MHZ/10 = 442.368 MHz)             -> ~576.5 MHz


In [19]:
# Capture and check against the three predictions
freq_500, spec_500 = acquire_spectrum(200, label='500 MHz aliasing test')

noise_floor = float(np.median(spec_500))
peak_idx    = int(np.argmax(spec_500))
peak_freq   = float(freq_500[peak_idx])
peak_snr    = float(spec_500[peak_idx]) - noise_floor

print(f"\n=== RESULT ===")
print(f"Peak found at: {peak_freq:.2f} MHz  (SNR {peak_snr:.1f} dB)")
print(f"\nDistance from each prediction:")
for name, pred in predictions.items():
    print(f"  {name:45s} -> {abs(peak_freq - pred):6.2f} MHz away")
print(f"\nClosest match: {min(predictions, key=lambda k: abs(peak_freq - predictions[k]))}")

if peak_snr < 6:
    print("\n*** WARNING: low/no SNR -- check the tone is reaching the ADC at all")
    print("    (could mean the analog chain is attenuating 500 MHz heavily if filter/LNA still in line)")

  Acquired 200 frames — 500 MHz aliasing test

=== RESULT ===
Peak found at: 0.00 MHz  (SNR 25.5 dB)

Distance from each prediction:
  H1: no decimation (code already correct)      -> 500.04 MHz away
  H2: 8x  (f_output = 552.96 MHz)               -> 423.63 MHz away
  H3: 10x (FS_MHZ/10 = 442.368 MHz)             -> 576.45 MHz away

Closest match: H2: 8x  (f_output = 552.96 MHz)


In [20]:
# ============================================================
# Plot the spectrum with all three hypothesis predictions marked
# Run this AFTER the capture/result cell above
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

colors = {'H1: no decimation (code already correct)': 'tab:green',
          'H2: 8x  (f_output = 552.96 MHz)': 'tab:orange',
          'H3: 10x (FS_MHZ/10 = 442.368 MHz)': 'tab:red'}

# --- Top: full 0-2200 MHz view (full Nyquist range of the labeled axis) ---
axes[0].plot(freq_500, spec_500, lw=0.7, color='#1f77b4')
for name, pred in predictions.items():
    axes[0].axvline(pred, color=colors[name], ls='--', lw=1.5, alpha=0.8,
                     label=f'{name}: {pred:.1f} MHz')
axes[0].axvline(peak_freq, color='black', ls='-', lw=1.2, alpha=0.6)
axes[0].set_xlabel('Frequency (MHz) -- current (uncorrected) labeling')
axes[0].set_ylabel('Power (dB)')
axes[0].set_title(f'500 MHz aliasing test -- full view  |  measured peak: {peak_freq:.2f} MHz')
axes[0].legend(loc='upper right', fontsize=9)
axes[0].grid(alpha=0.3)

# --- Bottom: zoomed view around whichever prediction is closest ---
closest_pred = predictions[min(predictions, key=lambda k: abs(peak_freq - predictions[k]))]
zoom_lo, zoom_hi = closest_pred - 30, closest_pred + 30
zoom_mask = (freq_500 >= zoom_lo) & (freq_500 <= zoom_hi)

axes[1].plot(freq_500[zoom_mask], spec_500[zoom_mask], lw=1.0, color='#1f77b4')
for name, pred in predictions.items():
    if zoom_lo <= pred <= zoom_hi:
        axes[1].axvline(pred, color=colors[name], ls='--', lw=1.5, alpha=0.8, label=name)
axes[1].axvline(peak_freq, color='black', ls='-', lw=1.5, alpha=0.7, label=f'measured peak: {peak_freq:.2f} MHz')
axes[1].set_xlabel('Frequency (MHz) -- current (uncorrected) labeling')
axes[1].set_ylabel('Power (dB)')
axes[1].set_title('Zoomed around closest-matching prediction')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
out_path = OUT_IQ / '500mhz_aliasing_test.png' if 'OUT_IQ' in dir() else Path('500mhz_aliasing_test.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {out_path}")

Saved: iq_observations/500mhz_aliasing_test.png


In [21]:
# Add this to the result cell, after peak_freq is computed
true_freq_inferred = peak_freq / 8.0
print(f"\nIf this matches the confirmed 8x scale factor:")
print(f"  Raw label says:      {peak_freq:.2f} MHz")
print(f"  True frequency (÷8): {true_freq_inferred:.2f} MHz  (expect ~52.96 MHz if 500 MHz tone aliased as predicted)")


If this matches the confirmed 8x scale factor:
  Raw label says:      0.00 MHz
  True frequency (÷8): 0.00 MHz  (expect ~52.96 MHz if 500 MHz tone aliased as predicted)


In [24]:
_cfg = soc.get_cfg()
soc.ddr4_buf.set_switch(_cfg['readouts'][ADC_CH]['avgbuf_fullpath'])
soc.clear_ddr4()
soc.ddr4_buf.arm(nt=NT_COARSE)
prog.acquire(soc, load_pulses=False, progress=False)
raw = soc.ddr4_buf.get_mem(nt=NT_COARSE)
I_samples = (raw[:, 0] if raw.ndim == 2 else raw).astype(np.float32)
I_samples = I_samples[:N_FFT]

print(f"Raw I samples (500 MHz test, ADC_CH={ADC_CH}):")
print(f"  Mean: {I_samples.mean():.4f}")
print(f"  Std:  {I_samples.std():.2f}")
print(f"  RMS:  {np.sqrt(np.mean(I_samples**2)):.2f}")

Raw I samples (500 MHz test, ADC_CH=0):
  Mean: -0.1558
  Std:  1.89
  RMS:  1.90


In [25]:
_cfg = soc.get_cfg()
soc.ddr4_buf.set_switch(_cfg['readouts'][ADC_CH]['avgbuf_fullpath'])
soc.clear_ddr4()
soc.ddr4_buf.arm(nt=NT_COARSE)
prog.acquire(soc, load_pulses=False, progress=False)
raw = soc.ddr4_buf.get_mem(nt=NT_COARSE)
I_samples = (raw[:, 0] if raw.ndim == 2 else raw).astype(np.float32)
I_samples = I_samples[:N_FFT]

print(f"Raw I samples (80 MHz reference check, ADC_CH={ADC_CH}):")
print(f"  Mean: {I_samples.mean():.4f}")
print(f"  Std:  {I_samples.std():.2f}")
print(f"  RMS:  {np.sqrt(np.mean(I_samples**2)):.2f}")

Raw I samples (80 MHz reference check, ADC_CH=0):
  Mean: 0.4286
  Std:  153.44
  RMS:  153.44
